# Day3_19 · 기상자료의 시간 패턴 확인

울산 일별 기상자료에서 이동평균, 월별 패턴, 특이한 날과 자기상관을 확인합니다.

## 이 노트북에서 확인할 내용

- 일별 값과 이동평균을 구분해 그릴 수 있습니다.
- 월 내부에서 특이한 기온을 찾을 수 있습니다.
- 시간자료의 가까운 행이 서로 독립적이지 않을 수 있음을 설명할 수 있습니다.

## 실행 방법

위에서 아래로 코드 셀을 한 개씩 실행하고 바로 다음 결과를 확인합니다. 수정 안내가 있는 곳에서는 표시된 값 하나만 바꾼 뒤 결과를 비교합니다.

### D3B-19-01 · 경로 도구 불러오기

In [ ]:
from pathlib import Path

### D3B-19-02 · 현재 폴더 확인

In [ ]:
current_folder = Path.cwd()

In [ ]:
current_folder

### D3B-19-03 · 데이터 폴더 후보 만들기

In [ ]:
data_candidates = [
    current_folder / 'data',
    current_folder.parent / 'data',
]

In [ ]:
data_candidates

### D3B-19-04 · 데이터 폴더 선택

In [ ]:
DATA_DIR = next(folder for folder in data_candidates if folder.exists())

In [ ]:
DATA_DIR

### D3B-19-05 · pandas 불러오기

In [ ]:
import pandas as pd

### D3B-19-06 · 그림 도구 불러오기

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

### D3B-19-07 · 울산 일별 기상자료 읽기

In [ ]:
weather = pd.read_csv(DATA_DIR / 'weather_daily.csv')

In [ ]:
weather.head()

### D3B-19-08 · 날짜를 날짜 형식으로 바꾸기

In [ ]:
weather['date'] = pd.to_datetime(weather['date'])

In [ ]:
weather[['date']].head()

### D3B-19-09 · 날짜 순서로 정렬하기

In [ ]:
weather = weather.sort_values('date').reset_index(drop=True)

In [ ]:
weather[['date', 'temperature_mean_c']].head()

### D3B-19-10 · 일별 평균기온 선그래프

In [ ]:
sns.lineplot(data=weather, x='date', y='temperature_mean_c', linewidth=1)
plt.ylabel('Mean temperature (°C)')
plt.show()

## 짧은 흔들림을 줄여 긴 변화를 보기

각 날짜와 앞선 날짜의 값을 함께 평균내면 하루 단위의 급한 흔들림이 줄어듭니다. 전문가들은 이를 이동평균이라고 부릅니다.

### D3B-19-11 · 7일 이동평균 만들기

In [ ]:
weather['temperature_7day_mean'] = weather['temperature_mean_c'].rolling(window=7, min_periods=1).mean()

In [ ]:
weather[['date', 'temperature_mean_c', 'temperature_7day_mean']].head(10).round(2)

### D3B-19-12 · 일별 기온과 7일 평균 함께 그리기

In [ ]:
plt.plot(weather['date'], weather['temperature_mean_c'], color='lightgray', linewidth=0.8, label='Daily')
plt.plot(weather['date'], weather['temperature_7day_mean'], color='red', linewidth=2, label='7-day mean')
plt.ylabel('Mean temperature (°C)')
plt.legend()
plt.show()

### D3B-19-13 · 월 열 만들기

In [ ]:
weather['month'] = weather['date'].dt.month

In [ ]:
weather[['date', 'month']].head()

### D3B-19-14 · 월별 기상 요약표 만들기

In [ ]:
monthly_weather = weather.groupby('month').agg(
    mean_temperature_c=('temperature_mean_c', 'mean'),
    max_temperature_c=('temperature_max_c', 'max'),
    precipitation_total_mm=('precipitation_mm', 'sum'),
    mean_humidity_percent=('humidity_percent', 'mean'),
)

In [ ]:
monthly_weather.round(2)

### D3B-19-15 · 월별 평균기온 그리기

In [ ]:
sns.lineplot(data=monthly_weather.reset_index(), x='month', y='mean_temperature_c', marker='o')
plt.xticks(range(1, 13))
plt.ylabel('Monthly mean temperature (°C)')
plt.show()

### D3B-19-16 · 월별 누적강수량 그리기

In [ ]:
sns.barplot(data=monthly_weather.reset_index(), x='month', y='precipitation_total_mm', color='steelblue')
plt.ylabel('Monthly precipitation total (mm)')
plt.show()

## 같은 달의 평균에서 많이 벗어난 날 찾기

### D3B-19-17 · 각 행에 해당 월의 평균기온 붙이기

In [ ]:
weather['month_temperature_mean'] = weather.groupby('month')['temperature_mean_c'].transform('mean')

In [ ]:
weather[['date', 'month', 'temperature_mean_c', 'month_temperature_mean']].head()

### D3B-19-18 · 월평균과의 차이 계산

In [ ]:
weather['temperature_anomaly_c'] = weather['temperature_mean_c'] - weather['month_temperature_mean']

In [ ]:
weather[['date', 'temperature_mean_c', 'temperature_anomaly_c']].head().round(2)

### D3B-19-19 · 월평균보다 특히 따뜻했던 날 찾기

In [ ]:
warm_anomaly_days = weather.nlargest(10, 'temperature_anomaly_c')

In [ ]:
warm_anomaly_days[['date', 'temperature_mean_c', 'month_temperature_mean', 'temperature_anomaly_c']].round(2)

### D3B-19-20 · 월평균보다 특히 추웠던 날 찾기

In [ ]:
cold_anomaly_days = weather.nsmallest(10, 'temperature_anomaly_c')

In [ ]:
cold_anomaly_days[['date', 'temperature_mean_c', 'month_temperature_mean', 'temperature_anomaly_c']].round(2)

### D3B-19-21 · 월평균과의 기온 차이 그리기

In [ ]:
colors = weather['temperature_anomaly_c'].ge(0).map({True: 'tomato', False: 'royalblue'})
plt.bar(weather['date'], weather['temperature_anomaly_c'], color=colors, width=1.0)
plt.axhline(0, color='black', linewidth=0.8)
plt.ylabel('Difference from monthly mean (°C)')
plt.show()

이 차이는 2024년 각 월의 내부 비교입니다. 여러 해의 장기 평년과 비교한 기후 이상값은 아닙니다.

### D3B-19-22 · 상위 5% 기온 기준 계산

In [ ]:
hot_threshold = weather['temperature_max_c'].quantile(0.95)

In [ ]:
hot_threshold

### D3B-19-23 · 상위 5%에 해당하는 더운 날 선택

In [ ]:
hot_days = weather.loc[weather['temperature_max_c'] >= hot_threshold]

In [ ]:
hot_days[['date', 'temperature_max_c', 'humidity_percent']].head(20).round(2)

### D3B-19-24 · 더운 날 수 확인

In [ ]:
len(hot_days)

## 강수량이 며칠 동안 누적된 모습 보기

### D3B-19-25 · 7일 누적강수량 만들기

In [ ]:
weather['precipitation_7day_total'] = weather['precipitation_mm'].rolling(window=7, min_periods=1).sum()

In [ ]:
weather[['date', 'precipitation_mm', 'precipitation_7day_total']].head(10).round(2)

### D3B-19-26 · 일별 강수량과 7일 누적값 그리기

In [ ]:
plt.bar(weather['date'], weather['precipitation_mm'], color='lightblue', width=1.0, label='Daily')
plt.plot(weather['date'], weather['precipitation_7day_total'], color='navy', linewidth=1.5, label='7-day total')
plt.ylabel('Precipitation (mm)')
plt.legend()
plt.show()

## 날짜 사이의 닮은 정도 확인하기

오늘 기온과 며칠 전 기온의 상관계수를 계산합니다. 전문가들은 이를 자기상관이라고 부릅니다.

### D3B-19-27 · 1·7·30일 자기상관 계산

In [ ]:
autocorrelation_table = pd.Series({
    '1 day': weather['temperature_mean_c'].autocorr(lag=1),
    '7 days': weather['temperature_mean_c'].autocorr(lag=7),
    '30 days': weather['temperature_mean_c'].autocorr(lag=30),
}, name='autocorrelation')

In [ ]:
autocorrelation_table.round(3)

자기상관이 크다는 것은 가까운 날짜의 값이 서로 비슷하다는 뜻입니다. 이런 자료를 무작위로 섞어 일반 K-fold를 적용하면 시간 구조가 평가자료에 섞일 수 있습니다.

### D3B-19-28 · 바꾸어 볼 이동평균 기간 정하기

In [ ]:
# 7을 14 또는 30으로 바꾸어 보세요.
rolling_days = 14

In [ ]:
rolling_days

### D3B-19-29 · 선택한 기간의 이동평균 계산

In [ ]:
chosen_rolling_temperature = weather['temperature_mean_c'].rolling(window=rolling_days, min_periods=1).mean()

In [ ]:
chosen_rolling_temperature.head(10).round(2)

### D3B-19-30 · 선택한 이동평균 그리기

In [ ]:
plt.plot(weather['date'], weather['temperature_mean_c'], color='lightgray', linewidth=0.7)
plt.plot(weather['date'], chosen_rolling_temperature, color='darkred', linewidth=2)
plt.title(f'{rolling_days}-day moving average')
plt.ylabel('Mean temperature (°C)')
plt.show()

### D3B-19-CODEX · Codex에 확장 분석 요청하기

In [ ]:
codex_request = '''weather DataFrame을 사용하세요. 기존 셀은 수정하지 마세요. humidity_percent의 14일 이동평균과 월평균 차이를 각각 새 열로 만들고, 두 값을 한 그림에 섞지 말고 별도 그래프 두 개로 제안해 주세요. 2024년 한 해만으로 장기 기후변화를 말하지 마세요.'''.strip()

In [ ]:
codex_request

### D3B-19-CHECK · 제안받은 코드를 실행하기 전에 확인하기

In [ ]:
codex_checklist = pd.Series(['날짜 순서가 유지되는가?', '이동평균과 월평균 차이를 구분하는가?', '한 해를 장기 기후추세로 해석하지 않는가?'], name='확인할 내용')

In [ ]:
codex_checklist